# 52 — Dynamic SBI Validation Results

Checks results from `scripts/validation/run_synth_sbi_dynamic.py` (cluster jobs).

**Goal:** Validate that dynamic SBI with RandomWalk priors can recover known
parameter trajectories from synthetic animals (10 BE + 10 SC).

**Key metrics:**
- Pearson/Spearman correlation between true and recovered trajectories
- MAE and NRMSE
- 95% CI calibration (should be ~0.95)
- PPC MSE: true model vs wrong model
- Model comparison accuracy (does PPC MSE correctly identify the true model?)

## 0 — Setup

In [ ]:
from shared_setup import *
import pickle
from scipy.stats import pearsonr, spearmanr
from pathlib import Path

In [ ]:
# Path to results — adjust if needed
SYNTH_DYNAMIC_DIR = VALIDATION_DIR / 'synth_sbi_dynamic'
print(f'Looking for results in: {SYNTH_DYNAMIC_DIR}')

if not SYNTH_DYNAMIC_DIR.exists():
    print('Directory not found. Check that:')
    print('  1. Cluster jobs have finished')
    print('  2. You ran scripts/validation/gather_synth_sbi_dynamic.py')
    print('  3. Results are synced to this machine')
else:
    pkl_files = sorted(SYNTH_DYNAMIC_DIR.glob('*.pkl'))
    print(f'Found {len(pkl_files)} files')
    for f in pkl_files:
        print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

## 1 — Load Results

In [ ]:
# Try loading the summary first (produced by gather script)
summary_path = SYNTH_DYNAMIC_DIR / 'summary.pkl'

if summary_path.exists():
    with open(summary_path, 'rb') as f:
        summary = pickle.load(f)
    results = summary['results']
    print(f'Loaded summary: {summary["n_correct"]}/{summary["n_total"]} correct')
else:
    # Load individual result files
    results = []
    for p in sorted(SYNTH_DYNAMIC_DIR.glob('*.pkl')):
        if p.name == 'summary.pkl':
            continue
        with open(p, 'rb') as f:
            data = pickle.load(f)
        if 'true_fit' in data:
            results.append(data)
        else:
            print(f'  Skipping {p.name}: old format')
    print(f'Loaded {len(results)} individual results')

if not results:
    raise RuntimeError('No results found. Run gather script first.')

In [ ]:
# Overview table
print(f'{"Animal":<12} {"True":<6} {"Correct?":<10} '
      f'{"PPC_true":<12} {"PPC_wrong":<12} {"Time(min)":<10}')
print('-' * 70)

for r in results:
    time_min = (
        r['true_fit']['training_time'] + r['wrong_fit']['training_time']
    ) / 60
    print(f'{r["animal_id"]:<12} {r["true_model"]:<6} '
          f'{"✓" if r["correct"] else "✗":<10} '
          f'{r["true_ppc_mse"]:<12.6f} {r["wrong_ppc_mse"]:<12.6f} '
          f'{time_min:<10.1f}')

## 2 — Model Comparison Accuracy

In [ ]:
n_total = len(results)
n_correct = sum(1 for r in results if r['correct'])
be_results = [r for r in results if r['true_model'] == 'BE']
sc_results = [r for r in results if r['true_model'] == 'SC']

be_correct = sum(1 for r in be_results if r['correct'])
sc_correct = sum(1 for r in sc_results if r['correct'])

print(f'Overall: {n_correct}/{n_total} ({100*n_correct/n_total:.0f}%)')
print(f'BE:      {be_correct}/{len(be_results)} '
      f'({100*be_correct/len(be_results):.0f}%)' if be_results else 'BE: none')
print(f'SC:      {sc_correct}/{len(sc_results)} '
      f'({100*sc_correct/len(sc_results):.0f}%)' if sc_results else 'SC: none')

In [ ]:
# PPC MSE: true model vs wrong model (violin/box plot)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, model_results, title in [
    (axes[0], be_results, 'True model: BE'),
    (axes[1], sc_results, 'True model: SC'),
]:
    if not model_results:
        ax.set_title(f'{title} — no data')
        continue

    true_mses = [r['true_ppc_mse'] for r in model_results]
    wrong_mses = [r['wrong_ppc_mse'] for r in model_results]

    positions = [0, 1]
    bp = ax.boxplot([true_mses, wrong_mses], positions=positions,
                     widths=0.4, patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][1].set_facecolor('darkorange')

    ax.set_xticks(positions)
    ax.set_xticklabels(['True model', 'Wrong model'])
    ax.set_ylabel('PPC MSE')
    ax.set_title(title)

fig.suptitle('Dynamic SBI: Model Comparison via PPC MSE', fontsize=12)
fig.tight_layout()
plt.show()

## 3 — Trajectory Recovery

In [ ]:
def compute_trajectory_metrics(result):
    """Compute recovery metrics for one animal."""
    true_fit = result['true_fit']
    trajectories = true_fit['trajectories']
    true_dicts = result['session_params_dicts']
    varying = true_fit['varying_params']
    n_sessions = result['n_sessions']

    metrics = {}
    for pname in varying:
        true_vals = np.array([d[pname] for d in true_dicts])
        traj = trajectories.get(pname)
        if traj is None or not isinstance(traj, dict):
            continue

        median = np.asarray(traj.get('median', []))
        ci_low = np.asarray(traj.get('ci_low', []))
        ci_high = np.asarray(traj.get('ci_high', []))

        if len(median) != n_sessions:
            continue

        mae = float(np.mean(np.abs(true_vals - median)))
        rmse = float(np.sqrt(np.mean((true_vals - median) ** 2)))

        if np.std(true_vals) > 1e-8 and np.std(median) > 1e-8:
            r, p = pearsonr(true_vals, median)
        else:
            r, p = np.nan, np.nan

        within_ci = float(np.mean(
            (true_vals >= ci_low) & (true_vals <= ci_high)))

        metrics[pname] = {
            'true': true_vals, 'median': median,
            'ci_low': ci_low, 'ci_high': ci_high,
            'mae': mae, 'rmse': rmse,
            'r': float(r), 'p': float(p),
            'ci_coverage': within_ci,
        }
    return metrics

In [ ]:
# Compute metrics for all animals
all_metrics = []
for r in results:
    m = compute_trajectory_metrics(r)
    m['_animal_id'] = r['animal_id']
    m['_model'] = r['true_model']
    m['_correct'] = r['correct']
    all_metrics.append(m)

# Summary table by model type
for model_type in ['BE', 'SC']:
    model_m = [m for m in all_metrics if m['_model'] == model_type]
    if not model_m:
        continue

    print(f'\n=== {model_type} ({len(model_m)} animals) ===')

    # Get parameter names from first valid result
    param_names = [
        k for k in model_m[0]
        if not k.startswith('_') and isinstance(model_m[0][k], dict)
    ]

    print(f'{"Param":<20} {"r (mean±sd)":<20} {"MAE":<14} '
          f'{"CI cov":<12} {"n":<4}')
    print('-' * 75)

    for pname in param_names:
        vals = [m[pname] for m in model_m if pname in m]
        if not vals:
            continue
        rs = [v['r'] for v in vals if not np.isnan(v['r'])]
        maes = [v['mae'] for v in vals]
        covs = [v['ci_coverage'] for v in vals]

        if rs:
            r_str = f'{np.mean(rs):.3f} ± {np.std(rs):.3f}'
        else:
            r_str = 'N/A'

        print(f'{pname:<20} {r_str:<20} {np.mean(maes):<14.4f} '
              f'{np.mean(covs):<12.2f} {len(vals):<4}')

In [ ]:
# Trajectory recovery plots: true vs recovered for each animal
for r, m in zip(results, all_metrics):
    param_names = [
        k for k in m
        if not k.startswith('_') and isinstance(m[k], dict)
    ]
    if not param_names:
        continue

    n_params = len(param_names)
    fig, axes = plt.subplots(1, n_params, figsize=(4.5 * n_params, 3.5))
    if n_params == 1:
        axes = [axes]

    for ax, pname in zip(axes, param_names):
        pm = m[pname]
        x = np.arange(len(pm['true']))

        ax.fill_between(x, pm['ci_low'], pm['ci_high'],
                        alpha=0.2, color='steelblue', label='95% CI')
        ax.plot(x, pm['median'], '-', color='steelblue',
                linewidth=1.5, label='Recovered')
        ax.plot(x, pm['true'], 'k--', linewidth=1.5, label='True')

        ax.set_xlabel('Session')
        ax.set_ylabel(pname)
        ax.set_title(f'r={pm["r"]:.3f}, CI cov={pm["ci_coverage"]:.2f}')
        ax.legend(fontsize=7)

    correct_str = '✓' if m['_correct'] else '✗'
    fig.suptitle(
        f'{m["_animal_id"]} (true: {m["_model"]}) {correct_str}',
        fontsize=12)
    fig.tight_layout()
    plt.show()

## 4 — Calibration Check

The 95% CI should contain the true value ~95% of the time.
If coverage is much lower, the posteriors are overconfident.
If much higher, they're too wide (underpowered but valid).

In [ ]:
# Calibration plot: coverage vs expected for each parameter
for model_type in ['BE', 'SC']:
    model_m = [m for m in all_metrics if m['_model'] == model_type]
    if not model_m:
        continue

    param_names = [
        k for k in model_m[0]
        if not k.startswith('_') and isinstance(model_m[0][k], dict)
    ]

    coverages = {}
    for pname in param_names:
        vals = [m[pname]['ci_coverage'] for m in model_m if pname in m]
        if vals:
            coverages[pname] = vals

    if not coverages:
        continue

    fig, ax = plt.subplots(figsize=(6, 4))
    positions = range(len(coverages))
    labels = list(coverages.keys())
    means = [np.mean(coverages[l]) for l in labels]

    ax.bar(positions, means, color='steelblue', alpha=0.7, edgecolor='white')
    for i, l in enumerate(labels):
        for v in coverages[l]:
            ax.scatter(i, v, c='black', s=15, alpha=0.3, zorder=3)

    ax.axhline(0.95, ls='--', color='red', alpha=0.5, label='Ideal (0.95)')
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('95% CI coverage')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'{model_type} — CI calibration')
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## 5 — Scatter: True vs Recovered (all sessions pooled)

In [ ]:
for model_type in ['BE', 'SC']:
    model_m = [m for m in all_metrics if m['_model'] == model_type]
    if not model_m:
        continue

    param_names = [
        k for k in model_m[0]
        if not k.startswith('_') and isinstance(model_m[0][k], dict)
    ]

    n_p = len(param_names)
    fig, axes = plt.subplots(1, n_p, figsize=(4.5 * n_p, 4))
    if n_p == 1:
        axes = [axes]

    for ax, pname in zip(axes, param_names):
        all_true, all_rec = [], []
        for m in model_m:
            if pname not in m or not isinstance(m[pname], dict):
                continue
            all_true.extend(m[pname]['true'])
            all_rec.extend(m[pname]['median'])

        all_true = np.array(all_true)
        all_rec = np.array(all_rec)

        ax.scatter(all_true, all_rec, s=10, alpha=0.3, c='steelblue')
        lims = [
            min(all_true.min(), all_rec.min()),
            max(all_true.max(), all_rec.max()),
        ]
        ax.plot(lims, lims, 'k--', alpha=0.5)
        ax.set_xlabel(f'True {pname}')
        ax.set_ylabel(f'Recovered {pname}')

        valid = ~np.isnan(all_true) & ~np.isnan(all_rec)
        if valid.sum() > 3:
            r, _ = pearsonr(all_true[valid], all_rec[valid])
            ax.set_title(f'{pname} (r={r:.3f}, n={valid.sum()})')

    fig.suptitle(f'{model_type} — True vs Recovered (all sessions)',
                 fontsize=12)
    fig.tight_layout()
    plt.show()